### Earthquake PINN Training on Colab GPU
This notebook clones the repository, installs dependencies, and runs the full-scale training.
It also includes optional Hyperparameter Optimization (Optuna).

**Note**: Make sure you have pushed your local changes (`git push origin dev`) before running this.

In [ ]:
!nvidia-smi

In [ ]:
# Install uv and clean up existing clone
!pip install uv
!rm -rf earthquake_proj
!git clone -b dev --single-branch https://github.com/sattary/earthquake_proj.git
%cd earthquake_proj

In [ ]:
# Sync environment
!uv sync

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/figs", exist_ok=True)

# Ensure package initialization for -m calls
if os.path.exists('src'): 
    if not os.path.exists('src/__init__.py'):
        open('src/__init__.py', 'a').close()
        print("Initialized 'src' as package.")
elif os.path.exists('earthquake_proj'):
    if not os.path.exists('earthquake_proj/__init__.py'):
        open('earthquake_proj/__init__.py', 'a').close()
        print("Initialized 'earthquake_proj' as package.")

In [ ]:
# Step 1: Tune Hyperparameters (Optional - skip if using defaults)
!if [ -d "src" ]; then PKG="src"; else PKG="earthquake_proj"; fi; \
export PYTHONPATH=$PYTHONPATH:$(pwd); \
print("Starting Hyperparameter Tuning with Optuna..."); \
uv run python -m $PKG.pinn.tune --trials 20 --epochs 200 --spatial-dim 3 --velocity-file data/Morteza_2023/Vel/Pwave.3D.txt

In [ ]:
# Step 2: Train with Best Params
!if [ -d "src" ]; then PKG="src"; else PKG="earthquake_proj"; fi; \
export PYTHONPATH=$PYTHONPATH:$(pwd); \
uv run python -m $PKG.pinn.cli train --epochs 10000 --n-coll 10000 --w-bc 10.0 --spatial-dim 3 --velocity-file data/Morteza_2023/Vel/Pwave.3D.txt --gpu

In [ ]:
# Generate Visualizations
!if [ -d "src" ]; then PKG="src"; else PKG="earthquake_proj"; fi; \
export PYTHONPATH=$PYTHONPATH:$(pwd); \
uv run python -m $PKG.pinn.visualize --model-path checkpoints/final_model.pth --depth 10.0 --output-stress results/figs/stress_map_10km.png --output-velocity results/figs/velocity_map_10km.png

In [ ]:
from IPython.display import Image, display
import os
for f in ['results/figs/stress_map_10km.png', 'results/figs/velocity_map_10km.png']:
    if os.path.exists(f): display(Image(f))

In [ ]:
# Pack Everything for Download/Active Storage
!zip -r results_archive.zip results/ checkpoints/final_model.pth
try:
  from google.colab import files
  files.download('results_archive.zip')
except ImportError:
  print('Not running on Google Colab, skipping download.')